# Test Scores Analysis
This notebook analyzes test scores from `test_scores.csv`. The metrics in the CSV are stored as string tuples formatted like `(score, weight)`, so this notebook parses them to calculate accurate weighted averages.

In [5]:
import pandas as pd
import ast

# Path to the data (relative to the eval/ folder)
file_path = "../exp_output/nova3r_img_cond_finetune_complete/nova3r_training/version_28/test_scores.csv"

# Load the data
df = pd.read_csv(file_path)
display(df.head())

,CD_16384_full,f_score_0.1_full,f_score_0.05_full,f_score_0.02_full,seq_name,id
0,"(0.09898930042982101, 5)","(0.6209800243377686, 5)","(0.37293002009391785, 5)","(0.1310698240995407, 5)",replica_pano_large_apartment_2_003,0
1,"(0.08038448542356491, 5)","(0.6895799040794373, 5)","(0.4203948974609375, 5)","(0.14641845226287842, 5)",replica_pano_large_apartment_2_003,5
2,"(0.10458593815565109, 5)","(0.5872719883918762, 5)","(0.369477242231369, 5)","(0.1316002458333969, 5)",replica_pano_large_apartment_2_003,10
3,"(0.08918973803520203, 5)","(0.6550797820091248, 5)","(0.39696961641311646, 5)","(0.14341238141059875, 5)",replica_pano_large_apartment_2_003,15
4,"(0.09276008605957031, 5)","(0.617182195186615, 5)","(0.3687193691730499, 5)","(0.1269565373659134, 5)",replica_pano_large_apartment_2_003,20


In [6]:
# Identify the metric columns (everything except seq_name and id)
metric_cols = [col for col in df.columns if col not in ['seq_name', 'id']]

def parse_metric(val_str):
    """Parses a string like '(0.07, 1)' into a Python tuple."""
    if pd.isna(val_str):
        return 0.0, 0
    if isinstance(val_str, str):
        try:
            return ast.literal_eval(val_str)
        except Exception:
            return 0.0, 0
    return 0.0, 0

def get_weighted_average(series):
    """Calculates the weighted average for a pandas Series containing tuple strings."""
    parsed = series.apply(parse_metric)
    scores = parsed.apply(lambda x: x[0])
    weights = parsed.apply(lambda x: x[1])
    
    total_weight = weights.sum()
    if total_weight > 0:
        return (scores * weights).sum() / total_weight
    return 0.0

def get_total_weight(series):
    """Calculates the sum of weights for a pandas Series containing tuple strings."""
    parsed = series.apply(parse_metric)
    weights = parsed.apply(lambda x: x[1])
    return weights.sum()

## Total Metrics
Calculations for the total combined weighted averages across all sequences.

In [7]:
print("--- Total Weighted Averages ---")
total_metrics = {}
for col in metric_cols:
    total_metrics[col] = get_weighted_average(df[col])
    print(f"{col}: {total_metrics[col]:.6f}")

--- Total Weighted Averages ---
CD_16384_full: 0.131589
f_score_0.1_full: 0.546620
f_score_0.05_full: 0.329224
f_score_0.02_full: 0.113436


## Per-Scene Metrics
Calculations for average metrics individually grouped by `seq_name`.

In [8]:
scene_results = []

for scene_name, group in df.groupby('seq_name'):
    row = {'seq_name': scene_name}
    # Sum the sample weights for the scene.
    row['num_samples'] = get_total_weight(group[metric_cols[0]])
    for col in metric_cols:
        row[col] = get_weighted_average(group[col])
    scene_results.append(row)

scene_df = pd.DataFrame(scene_results)
display(scene_df)

,seq_name,num_samples,CD_16384_full,f_score_0.1_full,f_score_0.05_full,f_score_0.02_full
0,replica_pano_hotel_0_000,100,0.215832,0.331130,0.164846,0.049000
1,replica_pano_large_apartment_0_004,100,0.089775,0.689163,0.453484,0.159680
2,replica_pano_large_apartment_2_001,100,0.162966,0.441149,0.258743,0.084090
3,replica_pano_large_apartment_2_002,100,0.092668,0.648723,0.390754,0.139683
4,replica_pano_large_apartment_2_003,100,0.096703,0.622935,0.378291,0.134726
